# BIAS RAG Law DB v3 — スコア駆動型抽象意味検索

## v2からの変更点
| v2 | v3 |
|---|---|
| テキストをベクトル化して類似検索 | 5軸スコアで抽象意味検索 |
| 論理強度スコアをランキング補正に使用 | スコアそのものが検索キー |
| claim_core / souten_pattern / weakness でDB構成 | スコア＋具体的事実のみでDB構成 |
| embedding modelが検索の主役 | LLMによるスコア化が検索の主役 |

## 設計思想
```
相談文（自然文）
  ↓ LLM：争いにおける自己の主張として抽象化
5軸スコア（抽象的な意味）
  ↓ ChromaDBでスコア距離計算
類似JSONを取得（具体的事実・証拠）
  ↓ LLM：γで参照度を制御して生成
有利/不利のアドバイス
```

## JSONの3層構造
```
【形式情報】  case_id / case_name / tachiba / subject
【検索スコア】shoko_sonzai / aite_hitei / jijitsu_meikaku / daisan_kaizai / horitsu_konkyo
【LLMへ渡す】yuri_facts / furi_facts / tsukkomi / statute
```

## フォルダ構成（Google Drive）
```
/MyDrive/
  case_*_genkoku.json   ← 原告型JSONデータ
  case_*_hikoku.json    ← 被告型JSONデータ
  bias_rag/
    chroma_db/          ← ChromaDB永続化（自動生成）
```

---
## Cell 0 — Google Drive マウント

マイドライブをColabにマウントし、JSONファイルの置き場所とChromaDBの保存先を設定します。

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# JSONメタデータの置き場所（マイドライブ直下）
METADATA_DIR = '/content/drive/MyDrive'

# ChromaDB保存先
DB_PATH = '/content/drive/MyDrive/bias_rag/chroma_db'
os.makedirs(DB_PATH, exist_ok=True)

print(f'メタデータ: {METADATA_DIR}')
print(f'DB保存先  : {DB_PATH}')

# 確認
jsons = [f for f in os.listdir(METADATA_DIR) if f.startswith('case_') and f.endswith('.json')]
print(f'JSONファイル数: {len(jsons)}')
for j in jsons:
    print(f'  {j}')

---
## Cell 1 — インストール

必要なライブラリをインストールします。
- `chromadb`: ベクトルDB（スコアをEmbeddingとして格納）
- `groq`: LLM API（相談文のスコア化・回答生成）

In [ ]:
%%capture
!pip install chromadb groq
print('インストール完了')

---
## Cell 2 — JSONメタデータ読み込み

マイドライブ直下の `case_*.json` を全件読み込みます。

各JSONは3層構造：
- **形式情報**: case_id, case_name, tachiba, subject
- **検索スコア**: shoko_sonzai, aite_hitei, jijitsu_meikaku, daisan_kaizai, horitsu_konkyo
- **LLMへ渡す**: yuri_facts, furi_facts, tsukkomi, statute

In [ ]:
import json, os

cases = []
for fname in sorted(os.listdir(METADATA_DIR)):
    if fname.startswith('case_') and fname.endswith('.json'):
        fpath = os.path.join(METADATA_DIR, fname)
        with open(fpath, encoding='utf-8') as f:
            data = json.load(f)
        cases.append(data)
        print(f'読込: {fname} — {data["tachiba"]} / {data["case_name"]}')

print(f'\n合計 {len(cases)} 件読み込み完了')

---
## Cell 3 — ChromaDB初期化 & スコアDB投入

v3の核心部分。

**v2との違い**: テキストをEmbeddingするのではなく、
**5軸スコアをそのままEmbedding（5次元ベクトル）としてChromaDBに投入**します。

これにより「スコアの近さ＝争いの構造の近さ」という抽象的な意味検索が実現します。

| キー | 意味 | 値域 |
|---|---|---|
| shoko_sonzai | 証拠の有無・強さ | 0.0〜1.0 |
| aite_hitei | 相手の否認度 | 0.0〜1.0 |
| jijitsu_meikaku | 事実の時系列明確さ | 0.0〜1.0 |
| daisan_kaizai | 第三者・代理人の介在 | 0.0〜1.0 |
| horitsu_konkyo | 法的根拠の強さ | 0.0〜1.0 |

In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path=DB_PATH)

# 既存コレクションがあれば再利用、なければ新規作成
collection = chroma_client.get_or_create_collection(
    name='bias_rag_v3',
    metadata={'description': 'スコア駆動型抽象意味検索 v3'}
)

# 5軸スコアをEmbeddingとして投入
SCORE_KEYS = ['shoko_sonzai', 'aite_hitei', 'jijitsu_meikaku', 'daisan_kaizai', 'horitsu_konkyo']

ids, embeddings_list, metadatas = [], [], []

for case in cases:
    # 5軸スコアを5次元ベクトルとして使用
    score_vec = [float(case.get(k, 0.5)) for k in SCORE_KEYS]

    ids.append(case['case_id'])
    embeddings_list.append(score_vec)
    metadatas.append({
        # 形式情報
        'case_id':   case['case_id'],
        'case_name': case['case_name'],
        'tachiba':   case['tachiba'],
        'subject':   case.get('subject', ''),
        # LLMへ渡す実質情報（JSON文字列で格納）
        'yuri_facts': json.dumps(case.get('yuri_facts', []), ensure_ascii=False),
        'furi_facts': json.dumps(case.get('furi_facts', []), ensure_ascii=False),
        'tsukkomi':   json.dumps(case.get('tsukkomi', []),   ensure_ascii=False),
        'statute':    json.dumps(case.get('statute', []),    ensure_ascii=False),
    })
    print(f'  ✓ {case["case_id"]} — スコア: {[round(s,2) for s in score_vec]}')

# upsert: 同じcase_idがあれば上書き、なければ追加
collection.upsert(
    ids=ids,
    embeddings=embeddings_list,
    metadatas=metadatas
)
print(f'\nDB投入完了。総件数: {collection.count()}')


---
## Cell 4 — LLM初期化（Groq）

Groq APIを使ってLLMを呼び出します。
2つの役割を担います：
1. **相談文 → 5軸スコアへの変換**（検索クエリの生成）
2. **ヒットしたJSONを使った回答生成**（アドバイス作成）

In [ ]:
from google.colab import userdata
import os, json, re
from groq import Groq

try:
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
except Exception:
    os.environ['GROQ_API_KEY'] = input('Groq APIキーを入力: ').strip()

client_llm = Groq(api_key=os.environ['GROQ_API_KEY'])

def call_llm(prompt, max_tokens=1500):
    resp = client_llm.chat.completions.create(
        model='llama-3.3-70b-versatile',
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=max_tokens,
        temperature=0.1
    )
    return resp.choices[0].message.content

def parse_json_response(response):
    text = re.sub(r'```json\s*|```\s*', '', response).strip()
    start = text.find('{')
    end   = text.rfind('}')
    if start == -1 or end == -1:
        raise ValueError(f'JSON not found: {text[:200]}')
    return json.loads(text[start:end+1])

test = call_llm('「こんにちは」とだけ返してください。')
print(f'Groq疎通テスト: {test[:50]}')
print('LLM: Groq / llama-3.3-70b-versatile — OK')

---
## Cell 5 — 相談文 → 5軸スコア変換

v3の核心その2。

相談者の入力を「争いにおける自己の主張」として読み取り、
LLMが5軸スコアに変換します。

このスコアがそのまま検索クエリの5次元ベクトルになります。

| 軸 | 評価基準 |
|---|---|
| shoko_sonzai | 主張を裏付ける証拠・書面・記録があるか |
| aite_hitei | 相手が事実を否定・争っているか |
| jijitsu_meikaku | いつ・誰が・何をしたかが明確か |
| daisan_kaizai | 代理人・仲介者・保証人等が介在しているか |
| horitsu_konkyo | 条文・契約書・判例等の法的根拠があるか |

In [ ]:
def query_to_scores(query_text):
    """相談文を5軸スコア（0.0〜1.0）に変換する"""
    prompt = f'''あなたは法的紛争を分析する専門家です。
以下の相談内容を「争いにおける自己の主張」として読み取り、
5つの軸でスコアリングしてください。

【相談内容】
{query_text}

【評価軸】
- shoko_sonzai: 主張を裏付ける証拠・書面・記録の有無・強さ（0.0=証拠なし / 1.0=証拠十分）
- aite_hitei: 相手方が事実を否定・争っている度合い（0.0=争いなし / 1.0=全面否認）
- jijitsu_meikaku: いつ・誰が・何をしたかの事実の明確さ（0.0=不明確 / 1.0=明確）
- daisan_kaizai: 代理人・仲介者・保証人等の第三者の介在（0.0=介在なし / 1.0=強く介在）
- horitsu_konkyo: 条文・契約書・判例等の法的根拠の強さ（0.0=根拠薄い / 1.0=根拠明確）

【出力形式】JSONのみ出力してください。
{{"shoko_sonzai": 0.0, "aite_hitei": 0.0, "jijitsu_meikaku": 0.0, "daisan_kaizai": 0.0, "horitsu_konkyo": 0.0, "tachiba": "請求する側 または 請求される側"}}'''
    return parse_json_response(call_llm(prompt, max_tokens=300))

print('query_to_scores 定義完了')

---
## Cell 6 — BIAS RAG 検索エンジン & 回答生成

### 検索
相談文から変換した5軸スコアをクエリベクトルとして、
ChromaDBのスコアベクトルと距離計算し、最も近い案件を取得します。

### 回答生成
ヒットしたJSONの `yuri_facts` / `furi_facts` / `tsukkomi` / `statute` をLLMに渡します。

γ（ガンマ）スライダーで参照度を制御：
- γ=0.0〜0.3: 事例の具体的事実をそのまま引用
- γ=0.4〜0.6: 相談内容に合わせて読み替え
- γ=0.7〜1.0: 争点の構造だけを抽象的に翻訳（ハルシネーション）

In [ ]:
SCORE_KEYS = ['shoko_sonzai', 'aite_hitei', 'jijitsu_meikaku', 'daisan_kaizai', 'horitsu_konkyo']

def bias_rag_search(query_text, k=3):
    """相談文をスコア化してChromaDBで類似検索"""
    print('スコア変換中...')
    scores = query_to_scores(query_text)
    print(f'  → {scores}')

    # 5軸スコアをクエリベクトルとして検索
    q_vec = [float(scores.get(k, 0.5)) for k in SCORE_KEYS]
    results = collection.query(
        query_embeddings=[q_vec],
        n_results=min(k, collection.count()),
        include=['metadatas', 'distances']
    )

    hits = []
    for meta, dist in zip(results['metadatas'][0], results['distances'][0]):
        similarity = max(0.0, 1.0 - dist / 2.0)
        hits.append({'meta': meta, 'similarity': similarity})

    return hits, scores


def generate_advice(query_text, hits, gamma=0.6):
    """ヒットしたJSONの具体的事実をLLMに渡して回答生成"""
    patterns = []
    for h in hits:
        meta = h['meta']
        patterns.append({
            'case_name':  meta.get('case_name', ''),
            'tachiba':    meta.get('tachiba', ''),
            'statute':    json.loads(meta.get('statute', '[]')),
            'yuri_facts': json.loads(meta.get('yuri_facts', '[]')),
            'furi_facts': json.loads(meta.get('furi_facts', '[]')),
            'tsukkomi':   json.loads(meta.get('tsukkomi', '[]')),
        })
    patterns_text = json.dumps(patterns, ensure_ascii=False, indent=2)

    # γで参照度を制御
    if gamma <= 0.3:
        t_inst = '参考事例の具体的事実・証拠・法条をそのまま活用して、相談内容に当てはめてください。'
    elif gamma <= 0.6:
        t_inst = '参考事例を「例」として参照しつつ、相談内容に合わせて読み替えてください。'
    else:
        t_inst = '参考事例の争点の構造と弱点パターンだけを抽象的に翻訳してください。具体的事実は引用せず、相談内容に創造的に適用してください。'

    prompt = f'''あなたは「紛争解決アドバイザー」です。
相談者の状況を整理し、有利な事実と不利な事実を並べます。
結論・勝訴予測は絶対に出しません。

【相談内容】
{query_text}

【参考事例】
{patterns_text}

【指示】{t_inst}

■ あなたに有利な事実・論点:
（箇条書きで3〜5点）

■ あなたに不利な事実・論点:
（箇条書きで3〜5点）

■ 今後確認すべき事実・証拠:
（箇条書きで2〜3点）

※ このアドバイスは情報提供のみです。法的判断は弁護士にご相談ください。'''

    return call_llm(prompt, max_tokens=1000)


def consult(query_text, gamma=0.6, k=2):
    print('=' * 60)
    print(f'【相談】{query_text}')
    print('=' * 60)
    hits, scores = bias_rag_search(query_text, k=k)
    print(f'\n類似案件 {len(hits)} 件取得:')
    for i, h in enumerate(hits):
        print(f"  [{i+1}] {h['meta']['case_name']} ({h['meta']['tachiba']}) 類似度={h['similarity']:.3f}")
    print(f'\n回答生成中（γ={gamma}）...')
    advice = generate_advice(query_text, hits, gamma=gamma)
    print('\n' + advice)
    return advice

print('アドバイザー定義完了')

---
## Cell 7 — テスト実行

3パターンでテストします：
1. 原告型に近い相談（γ=0.6）
2. 被告型に近い相談（γ=0.6）
3. 全く違う事例（γ=0.9 ハルシネーションモード）

In [ ]:
# テスト1: 原告型に近い相談
consult(
    "知人の紹介で土地を購入し、代金250万円を紹介者に支払いました。"
    "しかし売主が登記に応じてくれません。紹介者とも連絡が取れなくなりました。",
    gamma=0.6
)

In [ ]:
# テスト2: 被告型に近い相談
consult(
    "身に覚えのない契約書が届き、代金を払えと請求されています。"
    "契約書には私の実印が押されていますが、押した記憶がありません。",
    gamma=0.6
)

In [ ]:
# テスト3: ハルシネーションモード全開（γ=0.9）— 全く違う事例で類推
consult(
    "駅のホームで見知らぬ人にぶつかられて怪我をしました。"
    "相手は謝りもせず立ち去りました。損害賠償を請求できますか？",
    gamma=0.9
)

---
## Cell 8 — スライダーUI（インタラクティブ版）

γスライダーで「具体 ←→ 抽象」の参照度をリアルタイムに調整できます。

- **左端（0.0）**: 事例の具体的事実をそのまま引用
- **右端（1.0）**: 争点の構造だけを抽象的に翻訳（ハルシネーションモード）

In [ ]:
from ipywidgets import widgets
from IPython.display import display

query_box = widgets.Textarea(
    value='知人を通じて土地を購入し代金を支払いましたが、登記してもらえません。',
    description='相談内容:',
    layout=widgets.Layout(width='100%', height='80px')
)

gamma_slider = widgets.FloatSlider(
    value=0.5, min=0.0, max=1.0, step=0.1,
    description='具体 ← → 抽象:',
    style={'description_width': '130px'},
    readout_format='.1f'
)

run_button = widgets.Button(description='相談する', button_style='primary')
output     = widgets.Output()

def on_click(b):
    with output:
        output.clear_output()
        print(f'[設定] γ={gamma_slider.value}')
        consult(
            query_box.value,
            gamma=gamma_slider.value
        )

run_button.on_click(on_click)
display(query_box, gamma_slider, run_button, output)